In [71]:
!pip install crewai crewai-tools langchain-community -q

In [87]:
import os
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

In [88]:
from google.colab import userdata
DISCENTE_API_KEY = userdata.get('KEY_DISCENTE')
print('chave api carregada com sucesso')

chave api carregada com sucesso


In [89]:
# Configura o Gemini como LLM
gemini_llm = LLM(
    model="gemini/gemini-3.6-flash",
    api_key=DISCENTE_API_KEY  # Sua chave do Google
)

1. MENTAL FRAMEWORK: Como pensamos um agente

Todo agente no CrewAI tem 3 elementos essenciais:
- Role (Papel): QUEM ele é (ex: "Especialista em Suporte Técnico")
- Goal (Objetivo): O QUE ele busca (ex: "Resolver problemas técnicos rapidamente")
- Backstory (História): Contexto que molda o comportamento

In [90]:
# ============================================================
# 2. Ferramentas (Key elements of agent tools)
# ============================================================

# Ferramentas são funções que o agente pode executar
# Ex: buscar na web, ler documentação, calcular algo

os.environ["SERPER_API_KEY"] = DISCENTE_API_KEY

search_tool = SerperDevTool()  # Busca na internet
scrape_tool = ScrapeWebsiteTool()  # Lê conteúdo de sites

In [91]:
# ============================================================
# 3. CRIANDO OS AGENTES ESPECIALIZADOS
# ============================================================

# Agente 1: Atendimento ao Cliente
support_agent = Agent(
    role="Representante de Suporte Sênior",
    goal="Resolver dúvidas de clientes com empatia e precisão, "
         "buscando informações na base de conhecimento quando necessário",
    backstory=(
        "Você é um representante experiente de uma empresa de tecnologia. "
        "Você é conhecido por sua paciência e por sempre encontrar soluções. "
        "Você tem acesso a ferramentas de busca e documentação."
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    llm=gemini_llm,
    allow_delegation=True,  # Pode delegar para outros agentes
    memory=True,  # Lembra do contexto da conversa
)

# Agente 2: Garantia de Qualidade
qa_agent = Agent(
    role="Especialista em Qualidade de Suporte",
    goal="Revisar respostas do suporte para garantir que sejam "
         "completas, precisas e profissionais",
    backstory=(
        "Você é um revisor experiente que garante que toda comunicação "
        "com o cliente atenda aos padrões da empresa. Você identifica "
        "lacunas, imprecisões ou tons inadequados."
    ),
    verbose=True,
    llm=gemini_llm,
    allow_delegation=False,
)

# Agente 3: Coordenador (opcional - para mostrar hierarquia)
coordinator_agent = Agent(
    role="Coordenador de Atendimento",
    goal="Garantir que o cliente receba a melhor experiência possível, "
         "coordenando a equipe de suporte",
    backstory=(
        "Você supervisiona toda a operação de suporte. Você entende "
        "as necessidades do cliente e direciona para os especialistas certos."
    ),
    verbose=True,
    allow_delegation=True,
)

In [92]:
# ============================================================
# 4. DEFININDO AS TAREFAS
# ============================================================

# Tarefa 1: Resolver a dúvida do cliente
inquiry_task = Task(
    description=(
        "Um cliente entrou em contato com a seguinte dúvida: {customer_inquiry}\n\n"
        "1. Analise a dúvida do cliente\n"
        "2. Use as ferramentas disponíveis para buscar informações relevantes\n"
        "3. Elabore uma resposta clara e empática\n"
        "4. Se necessário, delegue para outro agente"
    ),
    expected_output=(
        "Uma resposta completa e profissional ao cliente, "
        "formatada em markdown, com tom amigável e solução clara"
    ),
    agent=support_agent,
)

# Tarefa 2: Revisar a resposta
qa_task = Task(
    description=(
        "Revise a resposta gerada pelo agente de suporte.\n\n"
        "1. Verifique se a resposta está completa\n"
        "2. Verifique se o tom é profissional e empático\n"
        "3. Aponte qualquer melhoria necessária\n"
        "4. Aprove ou solicite revisão"
    ),
    expected_output=(
        "Um relatório de QA com: status (Aprovado/Precisa Revisão), "
        "pontos fortes e sugestões de melhoria"
    ),
    agent=qa_agent,
    context=[inquiry_task],  # Depende da tarefa anterior
)

In [93]:
# ============================================================
# 5. MONTANDO A CREW (equipe)
# ============================================================

support_crew = Crew(
    agents=[support_agent, qa_agent],
    tasks=[inquiry_task, qa_task],
    process=Process.sequential,  # Executa em sequência
    verbose=True,
    memory=False,
)

In [94]:
# ============================================================
# 6. EXECUTANDO
# ============================================================
import nest_asyncio
import asyncio

nest_asyncio.apply()  # Permite reentrada no loop do Colab

print("=" * 60)
print("OLÁ! SISTEMA MULTI-AGENTE DE SUPORTE AO CLIENTE - EM QUE POSSO AJUDAR?")
print("=" * 60)

# Função assíncrona que executa a crew
async def rodar_crew():
    return await support_crew.kickoff_async(inputs={
        "customer_inquiry": (
            "Olá, comprei o curso 'Python para Dados' há 3 dias, "
            "mas não consigo acessar a área de membros. "
            "Aparece um erro '404 - Página não encontrada'. "
            "Já tentei limpar o cache e usar outro navegador. "
            "Podem me ajudar?"
        )
    })

# Executa a coroutine e ESPERA o resultado de verdade
result = asyncio.get_event_loop().run_until_complete(rodar_crew())

print("\n" + "=" * 60)
print("✅ RESULTADO FINAL")
print("=" * 60)
print(result)

OLÁ! SISTEMA MULTI-AGENTE DE SUPORTE AO CLIENTE - EM QUE POSSO AJUDAR?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 00ca0a0f-dfd5-407c-afe6-91d40f681596                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ❌ Memory Query Error ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Query Failed                                                                                            │
│  Source: Unified Memory                                                                                         │
│  Error: Memory requires an embedder for vector search but initialization failed: The CHROMA_OPENAI_API_KEY      │
│  environment variable is not set.                                                                               │
│                                                                                                                 │
│  To fix this, do one of the following:                                                                          │
│    - Set OPENAI_API_KEY for the default embedder (text-embedding-3-large)                                       │
│    - Pass a different embedder: Memory(embedder={{"provider": "google", "config": {{...}}}})                    │
│    - Pass a callable: Memory(embedder=my_embedding_function)                                                    │
│                                                                                                                 │
│  Docs: https://docs.crewai.com/concepts/memory                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Um cliente entrou em contato com a seguinte dúvida: Olá, comprei o curso 'Python para Dados' há 3 dias,  │
│  mas não consigo acessar a área de membros. Aparece um erro '404 - Página não encontrada'. Já tentei limpar o   │
│  cache e usar outro navegador. Podem me ajudar?                                                                 │
│                                                                                                                 │
│  1. Analise a dúvida do cliente                                                                                 │
│  2. Use as ferramentas disponíveis para buscar informações relevantes                                           │
│  3. Elabore uma resposta clara e empática                                                                       │
│  4. Se necessário, delegue para outro agente                                                                    │
│  ID: cee77c42-b89f-4d69-98a5-5359c92c46d6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Representante de Suporte Sênior                                                                         │
│                                                                                                                 │
│  Task: Um cliente entrou em contato com a seguinte dúvida: Olá, comprei o curso 'Python para Dados' há 3 dias,  │
│  mas não consigo acessar a área de membros. Aparece um erro '404 - Página não encontrada'. Já tentei limpar o   │
│  cache e usar outro navegador. Podem me ajudar?                                                                 │
│                                                                                                                 │
│  1. Analise a dúvida do cliente                                                                                 │
│  2. Use as ferramentas disponíveis para buscar informações relevantes                                           │
│  3. Elabore uma resposta clara e empática                                                                       │
│  4. Se necessário, delegue para outro agente                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Especialista em Qualidade de Suporte                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # RELATÓRIO DE AVALIAÇÃO DE QUALIDADE (QA)                                                                     │
│                                                                                                                 │
│  **Avaliador:** Especialista em Qualidade de Suporte                                                            │
│  **Data:** 24 de Maio de 2024                                                                                   │
│  **Item Avaliado:** Resposta ao Cliente — Chamado de Erro 404 (Acesso ao curso "Python para Dados")             │
│  **Status do Atendimento:** **APROVADO** (Com pequenas observações de edição)                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Resumo da Avaliação                                                                                     │
│                                                                                                                 │
│  A resposta gerada pelo agente de suporte atende com excelência aos padrões de qualidade exigidos. O agente     │
│  compreendeu perfeitamente o contexto do problema, validou as ações pregressas do cliente, explicou a causa     │
│  raiz de forma simples e forneceu um passo a passo claro para a resolução. O tom de voz é acolhedor,            │
│  profissional e altamente empático.                                                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Análise Detalhada por Critério                                                                          │
│                                                                                                                 │
│  | Critério | Avaliação | Análise |                                                                             │
│  | :--- | :---: | :--- |                                                                                        │
│  | **Completude** | **Excelente** | A resposta abordou todos os pontos necessários: confirmação da resolução,   │
│  explicação técnica simplificada, instruções de acesso e canal aberto para suporte contínuo. |                  │
│  | **Tom e Empatia** | **Excelente** | O tom é humanizado e empático. O agente validou a frustração do aluno e  │
│  expressou entusiasmo genuíno com o início dos estudos. |                                                       │
│  | **Precisão Técnica** | **Excelente** | O diagnóstico (falha de sincronização no LMS) foi coerente com os     │
│  testes prévios efetuados pelo usuário (cache/navegador). |                                                     │
│  | **Clareza e Formatação** | **Muito Boa** | Uso adequado de marcadores, negritos e estrutura visualmente      │
│  agradável e fácil de ler. |                           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Revise a resposta gerada pelo agente de suporte.                                                         │
│                                                                                                                 │
│  1. Verifique se a resposta está completa                                                                       │
│  2. Verifique se o tom é profissional e empático                                                                │
│  3. Aponte qualquer melhoria necessária                                                                         │
│  4. Aprove ou solicite revisão                                                                                  │
│  Agent: Especialista em Qualidade de Suporte                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ RESULTADO FINAL
# RELATÓRIO DE AVALIAÇÃO DE QUALIDADE (QA)

**Avaliador:** Especialista em Qualidade de Suporte  
**Data:** 24 de Maio de 2024  
**Item Avaliado:** Resposta ao Cliente — Chamado de Erro 404 (Acesso ao curso "Python para Dados")  
**Status do Atendimento:** **APROVADO** (Com pequenas observações de edição)

---

### 1. Resumo da Avaliação

A resposta gerada pelo agente de suporte atende com excelência aos padrões de qualidade exigidos. O agente compreendeu perfeitamente o contexto do problema, validou as ações pregressas do cliente, explicou a causa raiz de forma simples e forneceu um passo a passo claro para a resolução. O tom de voz é acolhedor, profissional e altamente empático.

---

### 2. Análise Detalhada por Critério

| Critério | Avaliação | Análise |
| :--- | :---: | :--- |
| **Completude** | **Excelente** | A resposta abordou todos os pontos necessários: confirmação da resolução, explicação técnica simplificada, instruções de acesso e canal aberto para supo

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 00ca0a0f-dfd5-407c-afe6-91d40f681596                                                                       │
│  Final Output: # RELATÓRIO DE AVALIAÇÃO DE QUALIDADE (QA)                                                       │
│                                                                                                                 │
│  **Avaliador:** Especialista em Qualidade de Suporte                                                            │
│  **Data:** 24 de Maio de 2024                                                                                   │
│  **Item Avaliado:** Resposta ao Cliente — Chamado de Erro 404 (Acesso ao curso "Python para Dados")             │
│  **Status do Atendimento:** **APROVADO** (Com pequenas observações de edição)                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Resumo da Avaliação                                                                                     │
│                                                                                                                 │
│  A resposta gerada pelo agente de suporte atende com excelência aos padrões de qualidade exigidos. O agente     │
│  compreendeu perfeitamente o contexto do problema, validou as ações pregressas do cliente, explicou a causa     │
│  raiz de forma simples e forneceu um passo a passo claro para a resolução. O tom de voz é acolhedor,            │
│  profissional e altamente empático.                                                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Análise Detalhada por Critério                                                                          │
│                                                                                                                 │
│  | Critério | Avaliação | Análise |                                                                             │
│  | :--- | :---: | :--- |                                                                                        │
│  | **Completude** | **Excelente** | A resposta abordou todos os pontos necessários: confirmação da resolução,   │
│  explicação técnica simplificada, instruções de acesso e canal aberto para suporte contínuo. |                  │
│  | **Tom e Empatia** | **Excelente** | O tom é humanizado e empático. O agente validou a frustração do aluno e  │
│  expressou entusiasmo genuíno com o início dos estudos. |                                                       │
│  | **Precisão Técnica** | **Excelente** | O diagnóstico (falha de sincronização no LMS) foi coerente com os     │
│  testes prévios efetuados pelo usuário (cache/navegador). |                                                     │
│  | **Clareza e Formatação** | **Muito Boa** | Uso adequado de marcadores, negritos e estrutura visualmente      │
│  agradável e fácil de ler. |                          